In [ ]:
# Parameters
eval_start = "2014-08-01"
eval_end = "2014-09-30"
dataset_choice = "imerg"
eval_vars = "total_precipitation_6hr"
apath = "/Datastorage/saptarishi.dhanuka_asp25/era5_data/era5_cache/"
params_path_old = '/Datastorage/saptarishi.dhanuka_asp25/gc_weights/origs/graphcast_1_13.npz'

params_path_new1 = '/Datastorage/saptarishi.dhanuka_asp25/gc_weights/graphcast_1_13_orig_2014-06-01_2014-07-30_FORECAST28_dynamic_weighing_india_mask_expt5.npz'
params_path_new2 = '/Datastorage/saptarishi.dhanuka_asp25/gc_weights/graphcast_1_13_orig_2014-06-01_2014-07-30_FORECAST28_dynamic_weighing_india_mask_expt3.npz'
norms_dir = '/Datastorage/saptarishi.dhanuka_asp25/gc_norms/'
plots_dir = 'plots/evals'
latmin, latmax, lonmin, lonmax = 6, 38, 35, 65
plot_timesteps = 7
output_pred_old_dir = '/Datastorage/saptarishi.dhanuka_asp25/preds_dir/'
output_pred_finetuned_dir = '/Datastorage/saptarishi.dhanuka_asp25/preds_dir/'


In [ ]:
# <eval_forecast.py>
"""
python /home/saptarishi.dhanuka_asp25/weather/graphcast_dir/graphcast/local_files/eval_forecast.py \ 
--eval_start "2024-08-01" \ 
--eval_end "2024-09-15" \ 
--eval_dataset_choice "imerg" \ 
--vars_to_eval "total_precipitation_6hr" \ 
--params_path_new1 "/Datastorage/saptarishi.dhanuka_asp25/gc_weights/graphcast_1_13_orig_2024-06-01_2024-09-15_FORECAST28.npz" \ 
--params_path_new2 "/Datastorage/saptarishi.dhanuka_asp25/gc_weights/graphcast_1_13_orig_2024-06-01_2024-07-30_FORECAST28.npz" \
--output_csv_path "./evaluation_results/forecast_mse.csv"
"""

"""
Complete evaluation of forecast against different datasets
"""
# <eval_forecast.py>
# Parameters
eval_start = "2024-08-01"
eval_end = "2024-09-05"
dataset_choice = "imerg"
eval_vars = "total_precipitation_6hr"
apath = "/Datastorage/saptarishi.dhanuka_asp25/era5_data/era5_cache/"
params_path_old = '/Datastorage/saptarishi.dhanuka_asp25/gc_weights/origs/graphcast_1_13.npz'

params_path_new1 = '/Datastorage/saptarishi.dhanuka_asp25/gc_weights/graphcast_1_13_orig_shapefile_2024-06-01_2024-07-30_FORECAST28_new.npz'
params_path_new2 = '/Datastorage/saptarishi.dhanuka_asp25/gc_weights/graphcast_1_13_orig_shapefile_2024-06-01_2024-07-30_FORECAST28.npz'
norms_dir = '/Datastorage/saptarishi.dhanuka_asp25/gc_norms/'
plots_dir = 'plots/evals'
latmin, latmax, lonmin, lonmax = 6, 38, 35, 65
plot_timesteps = 7
output_pred_old_dir = '/Datastorage/saptarishi.dhanuka_asp25/preds_dir/'
output_pred_finetuned_dir = '/Datastorage/saptarishi.dhanuka_asp25/preds_dir/'
region_vise = True
regions = ['Central_Northeast', 'Hilly_Regions', 'Northeast', 'Northwest', 'South_Peninsular', 'West_Central']

"""
Complete evaluation of forecast against different datasets with rainfall analysis
"""
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1"
import sys
import logging
import argparse
import dataclasses
import xarray as xr
import numpy as np
import pandas as pd
from datetime import datetime
from tqdm import tqdm
import time
import zarr
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from matplotlib.colors import LinearSegmentedColormap

import jax
import optax

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

# import utils
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '../..')))
from graphcast import checkpoint, data_utils, rollout, graphcast, normalization
import setup_jax_functions
from plotting import scale, select, plot_data, save_animation, save_static_plot, compute_difference_with_targets_sims, plot_sample_from_ds
from metrics import compute_rmse, compute_mae, compute_bias, compute_acc
from utils import regrid_hres_fine_to_coarse, generate_sample_era5_dataset, grads_fn, parse_args, process_to_graphcast_format, compute_mse, compute_mse_diffs, mask_dbase_india_buffer, mask_dbase_regions

sys.path.append('/home/saptarishi.dhanuka_asp25/weather/graphcast_dir/gc_dist')
import trainer.dataloader
from dist_utils import construct_era5_imerg
from datetime import datetime

In [ ]:
# <eval_forecast.py>
"""
python /home/saptarishi.dhanuka_asp25/weather/graphcast_dir/graphcast/local_files/eval_forecast.py \ 
--eval_start "2024-08-01" \ 
--eval_end "2024-09-15" \ 
--eval_dataset_choice "imerg" \ 
--vars_to_eval "total_precipitation_6hr" \ 
--params_path_new1 "/Datastorage/saptarishi.dhanuka_asp25/gc_weights/graphcast_1_13_orig_2024-06-01_2024-09-15_FORECAST28.npz" \ 
--params_path_new2 "/Datastorage/saptarishi.dhanuka_asp25/gc_weights/graphcast_1_13_orig_2024-06-01_2024-07-30_FORECAST28.npz" \
--output_csv_path "./evaluation_results/forecast_mse.csv"
"""

"""
Complete evaluation of forecast against different datasets
"""
import os
import sys
import logging
import argparse
import dataclasses
import xarray as xr
import numpy as np
import pandas as pd
from datetime import datetime
from tqdm import tqdm
import time
import zarr
import matplotlib.pyplot as plt
import cartopy.crs as ccrs

import jax
import optax

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))
from graphcast import checkpoint, data_utils, rollout, graphcast, normalization
import save_params_utils
import setup_jax_functions
from plotting import scale, select, plot_data, save_animation, save_static_plot, compute_difference_with_targets_sims, plot_sample_from_ds
from metrics import compute_rmse, compute_mae, compute_bias, compute_acc
from utils import regrid_hres_fine_to_coarse, generate_sample_era5_dataset, grads_fn, parse_args, process_to_graphcast_format, compute_mse, compute_mse_diffs


sys.path.append('/home/saptarishi.dhanuka_asp25/weather/graphcast_dir/gc_dist')
import trainer.dataloader
from dist_utils import construct_era5_imerg

# --- Basic Setup ---
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
# args = parse_args()

# Add new argument to the parser in utils.py
# For now, let's handle it here if it's not in the original file

output_csv_path = "./forecast_evaluation_mse.csv"

# eval_start = args.eval_start
# eval_end = args.eval_end
# dataset_choice = args.eval_dataset_choice
# eval_vars = args.vars_to_eval
# apath = args.eval_data_path
# params_path_old = args.params_path_old
# params_path_new1 = args.params_path_new1
# params_path_new2 = args.params_path_new2

# --- Data and Model Loading ---
logging.info(f"Loading dataset '{dataset_choice}' from {eval_start} to {eval_end}")
if dataset_choice == "imerg":
    apath = '/Datastorage/saptarishi.dhanuka_asp25/era5_data/era5_cache/'
    dbase,_ = trainer.dataloader.open_databases(apath,None)
    dbase = construct_era5_imerg(dbase)
    # Load a slightly larger window to ensure we have the day before the start_date for initialization
    load_start_date = pd.to_datetime(eval_start) - pd.Timedelta(days=1)
    eval_time_ds = dbase.sel(time=slice(load_start_date.strftime('%Y-%m-%d'), eval_end))
    del dbase

select_time_eval = process_to_graphcast_format(eval_time_ds)
logging.info(f"Full data range loaded: {select_time_eval.time.values[0]} to {select_time_eval.time.values[-1]}")

logging.info("Loading models and normalization stats...")
with open(params_path_old, 'rb') as f:
    ckpt = checkpoint.load(f, graphcast.CheckPoint)
params = ckpt.params

with open(params_path_new1, "rb") as f:
    new_params1 = checkpoint.load(f, graphcast.CheckPoint).params
with open(params_path_new2, "rb") as f:
    new_params2 = checkpoint.load(f, graphcast.CheckPoint).params

with open('/Datastorage/saptarishi.dhanuka_asp25/gc_norms/diffs_stddev_by_level.nc', 'rb') as f:
    diffs_stddev_by_level = xr.load_dataset(f).compute()
with open('/Datastorage/saptarishi.dhanuka_asp25/gc_norms/stddev_by_level.nc', 'rb') as f:
    stddev_by_level = xr.load_dataset(f).compute()
with open('/Datastorage/saptarishi.dhanuka_asp25/gc_norms/mean_by_level.nc', 'rb') as f:
    mean_by_level = xr.load_dataset(f).compute()

# --- JAX Function Setup ---
logging.info("Setting up JAX functions...")
state = {}
model_config = ckpt.model_config
task_config = ckpt.task_config
setup_jax_functions.update_configs({
    'params': ckpt.params, 'state': state, 'model_config': ckpt.model_config, 'task_config': task_config,
    'mean_by_level': mean_by_level, 'stddev_by_level': stddev_by_level, 'diffs_stddev_by_level': diffs_stddev_by_level
})
run_forward_jitted = setup_jax_functions.drop_state(setup_jax_functions.with_params(jax.jit(setup_jax_functions.with_configs(
    setup_jax_functions.run_forward.apply))))
jax.config.update("jax_enable_x64", True)

def run_model(params, state, inputs, targets_template, forcings):
    return run_forward_jitted(
        rng=jax.random.PRNGKey(0),
        inputs=inputs,
        targets_template=targets_template,
        forcings=forcings,
        params=params,
        state=state
    )

# --- HRES Data Handling ---
import glob
def extract_hres_date(path):
    parts = path.split('_')
    # print(parts)
    year = 2024  # fixed, or extract if you want dynamic
    month = int(parts[5])
    day = int(parts[6])
    return datetime(year, month, day)

logging.info("Mapping HRES forecast files...")
hres_file_list = glob.glob("/Datastorage/saptarishi.dhanuka_asp25/forecasts_hres/raw_hres/2024/*.grib")
hres_files_map = {extract_hres_date(p): p for p in hres_file_list}

# =============================================================================
# === NEW SYSTEMATIC EVALUATION AND VERIFICATION PROCEDURE ====================
# =============================================================================
all_results = []
initialization_dates = pd.to_datetime(pd.date_range(start=eval_start, end=eval_end, freq='D'))



2025-07-15 21:23:33,253 - INFO - Loading dataset 'imerg' from 2024-08-01 to 2024-10-15


Found ['/Datastorage/saptarishi.dhanuka_asp25/era5_data/era5_cache/2023/01', '/Datastorage/saptarishi.dhanuka_asp25/era5_data/era5_cache/2023/02', '/Datastorage/saptarishi.dhanuka_asp25/era5_data/era5_cache/2023/03', '/Datastorage/saptarishi.dhanuka_asp25/era5_data/era5_cache/2023/04', '/Datastorage/saptarishi.dhanuka_asp25/era5_data/era5_cache/2023/05', '/Datastorage/saptarishi.dhanuka_asp25/era5_data/era5_cache/2023/06', '/Datastorage/saptarishi.dhanuka_asp25/era5_data/era5_cache/2023/07', '/Datastorage/saptarishi.dhanuka_asp25/era5_data/era5_cache/2023/08', '/Datastorage/saptarishi.dhanuka_asp25/era5_data/era5_cache/2023/09', '/Datastorage/saptarishi.dhanuka_asp25/era5_data/era5_cache/2023/10', '/Datastorage/saptarishi.dhanuka_asp25/era5_data/era5_cache/2023/11', '/Datastorage/saptarishi.dhanuka_asp25/era5_data/era5_cache/2023/12', '/Datastorage/saptarishi.dhanuka_asp25/era5_data/era5_cache/2024/01', '/Datastorage/saptarishi.dhanuka_asp25/era5_data/era5_cache/2024/02', '/Datastorage

Looping through era5 days for conversion:  40%|███▉      | 348/871 [00:00<00:00, 1739.06it/s]

Info: used IMERG data from 2024-01-01 00:00:00 for target Timestamp('2023-12-25 00:00:00')
Info: used IMERG data from 2024-01-01 00:00:00 for target Timestamp('2023-12-26 00:00:00')
Info: used IMERG data from 2024-01-01 00:00:00 for target Timestamp('2023-12-27 00:00:00')
Info: used IMERG data from 2024-01-01 00:00:00 for target Timestamp('2023-12-28 00:00:00')
Info: used IMERG data from 2024-01-01 00:00:00 for target Timestamp('2023-12-29 00:00:00')
Info: used IMERG data from 2024-01-01 00:00:00 for target Timestamp('2023-12-30 00:00:00')
Info: used IMERG data from 2024-01-01 00:00:00 for target Timestamp('2023-12-31 00:00:00')


Looping through era5 days for conversion:  83%|████████▎ | 720/871 [00:06<00:01, 106.05it/s] 


Exceeded IMERG timespan, stopped replacing with IMERG
Skipping 2023-01-01 00:00:00: no IMERG data for 2023-01-01 00:00:00
Skipping 2023-01-01 06:00:00: no IMERG data for 2023-01-01 00:00:00
Skipping 2023-01-01 12:00:00: no IMERG data for 2023-01-01 00:00:00
Skipping 2023-01-01 18:00:00: no IMERG data for 2023-01-01 00:00:00
Skipping 2023-01-02 00:00:00: no IMERG data for 2023-01-02 00:00:00
Skipping 2023-01-02 06:00:00: no IMERG data for 2023-01-02 00:00:00
Skipping 2023-01-02 12:00:00: no IMERG data for 2023-01-02 00:00:00
Skipping 2023-01-02 18:00:00: no IMERG data for 2023-01-02 00:00:00
Skipping 2023-01-03 00:00:00: no IMERG data for 2023-01-03 00:00:00
Skipping 2023-01-03 06:00:00: no IMERG data for 2023-01-03 00:00:00
Skipping 2023-01-03 12:00:00: no IMERG data for 2023-01-03 00:00:00
Skipping 2023-01-03 18:00:00: no IMERG data for 2023-01-03 00:00:00
Skipping 2023-01-04 00:00:00: no IMERG data for 2023-01-04 00:00:00
Skipping 2023-01-04 06:00:00: no IMERG data for 2023-01-04 00:

2025-07-15 21:23:50,692 - INFO - Full data range loaded: 0 nanoseconds to 6631200000000000 nanoseconds
2025-07-15 21:23:50,693 - INFO - Loading models and normalization stats...
2025-07-15 21:23:51,110 - INFO - Setting up JAX functions...
2025-07-15 21:23:51,111 - INFO - Mapping HRES forecast files...


In [121]:
# Max forecast length is 7 days (28 steps of 6 hours)
MAX_FORECAST_STEPS = 28
target_lead_times_str = f"{(MAX_FORECAST_STEPS) * 6}h" # +1 to be safe with slicing
target_lead_times_slice = slice("6h", target_lead_times_str)
forecast_horizons_def = {1: 4, 3: 12, 7: 28} # In 6-hourly steps

logging.info(f"Starting evaluation for {len(initialization_dates)} initialization dates.")


2025-07-15 22:43:47,290 - INFO - Starting evaluation for 76 initialization dates.


In [132]:
forecast_horizons_def = {i: i for i in range(0,168, 6)}

In [4]:
len(initialization_dates)

76

In [47]:
init_date = initialization_dates[0]
start_slice = init_date - pd.Timedelta(hours=6)
end_slice = init_date
start_slice, end_slice, type(start_slice)

(Timestamp('2024-07-31 18:00:00'),
 Timestamp('2024-08-01 00:00:00'),
 pandas._libs.tslibs.timestamps.Timestamp)

In [48]:
end_slice = init_date + pd.Timedelta(days=7)
end_slice

Timestamp('2024-08-08 00:00:00')

In [49]:
print(select_time_eval.datetime.values[0][0])
offset_end = end_slice - select_time_eval.datetime.values[0][0]
offset_start = start_slice - select_time_eval.datetime.values[0][0]
offset_start, offset_end

2024-07-31T00:00:00.000000000


(Timedelta('0 days 18:00:00'), Timedelta('8 days 00:00:00'))

In [50]:
from dask.diagnostics import ProgressBar
with ProgressBar():
    eval_sim_data = select_time_eval.sel(time=slice(offset_start, offset_end)).compute()

[########################################] | 100% Completed | 1.71 sms


In [51]:
target_lead_times_slice

slice('6h', '168h', None)

In [52]:
eval_inputs, eval_targets, eval_forcings = data_utils.extract_inputs_targets_forcings(
    eval_sim_data, target_lead_times=target_lead_times_slice, **dataclasses.asdict(task_config))

# Ensure we have enough target data for the longest forecast
if eval_targets.sizes['time'] < MAX_FORECAST_STEPS:
    logging.warning(f"Not enough target data for a 7-day forecast from {init_date}. Have {eval_targets.sizes['time']} steps. Skipping.")


targets_template = eval_targets * np.nan
ground_truth_var = eval_targets[eval_vars]


In [53]:
print("Eval Inputs:   ", eval_inputs.dims.mapping)
print("Eval Targets:  ", eval_targets.dims.mapping)
print("Eval Forcings: ", eval_forcings.dims.mapping)

Eval Inputs:    {'batch': 1, 'time': 2, 'lat': 181, 'lon': 360, 'level': 13}
Eval Targets:   {'batch': 1, 'time': 28, 'lat': 181, 'lon': 360, 'level': 13}
Eval Forcings:  {'batch': 1, 'time': 28, 'lat': 181, 'lon': 360}


In [54]:

# 3. Run all Graphcast models
logging.debug(f"Running Graphcast models for {init_date}")

predictions_base = run_model(params, state, eval_inputs, targets_template, eval_forcings)
print("Predictions Old: ", predictions_base.dims.mapping)

Predictions Old:  {'time': 28, 'batch': 1, 'lat': 181, 'lon': 360, 'level': 13}


In [55]:

predictions_ft1 = run_model(new_params1, state, eval_inputs, targets_template, eval_forcings)
print("Predictions Finetuned 1: ", predictions_ft1.dims.mapping)

predictions_ft2 = run_model(new_params2, state, eval_inputs, targets_template, eval_forcings)
print("Predictions Finetuned 2: ", predictions_ft2.dims.mapping)


models_to_eval = {
    'Graphcast_Base': predictions_base,
    'Graphcast_Finetuned1': predictions_ft1,
    'Graphcast_Finetuned2': predictions_ft2,
}

Predictions Finetuned 1:  {'time': 28, 'batch': 1, 'lat': 181, 'lon': 360, 'level': 13}
Predictions Finetuned 2:  {'time': 28, 'batch': 1, 'lat': 181, 'lon': 360, 'level': 13}


In [ ]:
# 4. Load, regrid, and align HRES forecast for the same initialization date
hres_predictions = None
hres_file_path = hres_files_map.get(init_date.to_pydatetime().replace(hour=0, minute=0, second=0, microsecond=0))


/home/saptarishi.dhanuka_asp25/.conda/envs/graphcast_fine/lib/python3.11/site-packages/cfgrib/xarray_plugin.py:131: FutureWarning: In a future version of xarray decode_timedelta will default to False rather than None. To silence this warning, set decode_timedelta to True, False, or a 'CFTimedeltaCoder' instance.
  vars, attrs, coord_names = xr.conventions.decode_cf_variables(


Regridding tp from high resolution to coarse resolution 1.0 degrees
Regridder built


2025-07-15 21:59:55,386 - WARNING - Failed to process HRES file /Datastorage/saptarishi.dhanuka_asp25/forecasts_hres/raw_hres/2024/hres_2024_8_1_ppt_6hourly.grib: the new name 'time' conflicts
MSE Calc: 100%|██████████| 3/3 [00:00<00:00, 53.07it/s]


In [60]:
hres_ds= xr.open_dataset(hres_file_path, engine='cfgrib')

/home/saptarishi.dhanuka_asp25/.conda/envs/graphcast_fine/lib/python3.11/site-packages/cfgrib/xarray_plugin.py:131: FutureWarning: In a future version of xarray decode_timedelta will default to False rather than None. To silence this warning, set decode_timedelta to True, False, or a 'CFTimedeltaCoder' instance.
  vars, attrs, coord_names = xr.conventions.decode_cf_variables(


In [71]:
hres_regridded = regrid_hres_fine_to_coarse(hres_ds, variable='tp', coarse_resolution=1.0)

Regridding tp from high resolution to coarse resolution 1.0 degrees
Regridder built


In [81]:
hres_regrid_og = hres_regridded.copy(deep=True)

In [82]:
print(hres_regrid_og)

<xarray.DataArray 'tp' (step: 29, lat: 181, lon: 360)> Size: 8MB
array([[[0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
         0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
        [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
         0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
        [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
         0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
        ...,
        [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
         0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
        [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
         0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
        [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
         0.00000000e+00, 0.00000000e+00, 0.00000000e+00]],

       [[2.28881836e-05, 2.28881836e-05, 2.28881836e-05, ...,
         2.28881836e-05, 2.28881836e-05, 2.28881836e-05],
        [8.77380371e-05, 8.77380371e-05, 8.77380371e-05, ...,
         6.8664550

In [83]:
hres_predictions = hres_regridded.drop({'time'}).rename({'step': 'time'}).isel(time=slice(1,None)).assign_coords(time=eval_targets.time)
hres_predictions

/tmp/ipykernel_1131528/1159762928.py:1: DeprecationWarning: dropping variables using `drop` is deprecated; use drop_vars.
  hres_predictions = hres_regridded.drop({'time'}).rename({'step': 'time'}).isel(time=slice(1,None)).assign_coords(time=eval_targets.time)


<xarray.DataArray 'tp' (time: 28, lat: 181, lon: 360)> Size: 7MB
array([[[2.28881836e-05, 2.28881836e-05, 2.28881836e-05, ...,
         2.28881836e-05, 2.28881836e-05, 2.28881836e-05],
        [8.77380371e-05, 8.77380371e-05, 8.77380371e-05, ...,
         6.86645508e-05, 8.77380371e-05, 8.77380371e-05],
        [2.36511230e-04, 2.36511230e-04, 2.47955322e-04, ...,
         2.25067139e-04, 2.25067139e-04, 2.36511230e-04],
        ...,
        [4.19616699e-05, 4.19616699e-05, 3.81469727e-05, ...,
         3.81469727e-05, 3.81469727e-05, 4.19616699e-05],
        [1.25885010e-04, 1.25885010e-04, 1.25885010e-04, ...,
         1.18255615e-04, 1.25885010e-04, 1.25885010e-04],
        [2.17437744e-04, 2.17437744e-04, 2.17437744e-04, ...,
         2.17437744e-04, 2.17437744e-04, 2.17437744e-04]],

       [[3.43322754e-05, 3.43322754e-05, 3.43322754e-05, ...,
         3.43322754e-05, 3.43322754e-05, 3.43322754e-05],
        [1.56402588e-04, 1.56402588e-04, 1.56402588e-04, ...,
         1.33514404e-04, 1.56402588e-04, 1.56402588e-04],
        [3.89099121e-04, 3.89099121e-04, 4.08172607e-04, ...,
         3.77655029e-04, 3.77655029e-04, 3.89099121e-04],
...
        [9.53674316e-03, 9.53674316e-03, 1.01470947e-02, ...,
         9.29260254e-03, 9.29260254e-03, 9.53674316e-03],
        [1.38702393e-02, 1.38702393e-02, 1.38702393e-02, ...,
         1.36566162e-02, 1.38702393e-02, 1.38702393e-02],
        [1.06658936e-02, 1.06658936e-02, 1.06658936e-02, ...,
         1.06658936e-02, 1.06658936e-02, 1.06658936e-02]],

       [[4.88281250e-04, 4.88281250e-04, 4.88281250e-04, ...,
         4.88281250e-04, 4.88281250e-04, 4.88281250e-04],
        [5.64575195e-04, 5.64575195e-04, 5.64575195e-04, ...,
         5.49316406e-04, 5.64575195e-04, 5.64575195e-04],
        [7.17163086e-04, 7.17163086e-04, 7.47680664e-04, ...,
         6.86645508e-04, 6.86645508e-04, 7.17163086e-04],
        ...,
        [9.56726074e-03, 9.56726074e-03, 1.01623535e-02, ...,
         9.30786133e-03, 9.30786133e-03, 9.56726074e-03],
        [1.39617920e-02, 1.39617920e-02, 1.39617920e-02, ...,
         1.37634277e-02, 1.39617920e-02, 1.39617920e-02],
        [1.07727051e-02, 1.07727051e-02, 1.07727051e-02, ...,
         1.07727051e-02, 1.07727051e-02, 1.07727051e-02]]], dtype=float32)
Coordinates:
    number      int64 8B 0
    surface     float64 8B 0.0
    valid_time  (time) datetime64[ns] 224B 2024-08-01T06:00:00 ... 2024-08-08
  * lat         (lat) float64 1kB -90.0 -89.0 -88.0 -87.0 ... 88.0 89.0 90.0
  * lon         (lon) float64 3kB 0.0 1.0 2.0 3.0 ... 356.0 357.0 358.0 359.0
  * time        (time) timedelta64[ns] 224B 0 days 06:00:00 ... 7 days 00:00:00
    expver      (time) <U4 448B '0001' '0001' '0001' ... '0001' '0001' '0001'
Attributes: (12/23)
    GRIB_paramId:                    228
    GRIB_dataType:                   fc
    GRIB_numberOfPoints:             6599680
    GRIB_typeOfLevel:                surface
    GRIB_stepUnits:                  1
    GRIB_stepType:                   instant
    ...                              ...
    GRIB_totalNumber:                0
    GRIB_units:                      m
    long_name:                       Total precipitation
    units:                           m
    standard_name:                   unknown
    regrid_method:                   nearest_s2d

In [93]:
if hres_file_path:
    try:
        logging.debug(f"Processing HRES file: {hres_file_path}")
        hres_ds = xr.open_dataset(hres_file_path, engine='cfgrib')
        hres_regridded = regrid_hres_fine_to_coarse(hres_ds, variable='tp', coarse_resolution=1.0)
        # Align time dimension with Graphcast targets
        hres_predictions = hres_regridded.drop_vars({'time'}).rename({'step': 'time'}).isel(time=slice(1,None)).assign_coords(time=eval_targets.time)
        models_to_eval['HRES'] = hres_predictions
    except Exception as e:
        logging.warning(f"Failed to process HRES file {hres_file_path}: {e}")
else:
    logging.warning(f"No HRES file found for init date {init_date}")


/home/saptarishi.dhanuka_asp25/.conda/envs/graphcast_fine/lib/python3.11/site-packages/cfgrib/xarray_plugin.py:131: FutureWarning: In a future version of xarray decode_timedelta will default to False rather than None. To silence this warning, set decode_timedelta to True, False, or a 'CFTimedeltaCoder' instance.
  vars, attrs, coord_names = xr.conventions.decode_cf_variables(


Regridding tp from high resolution to coarse resolution 1.0 degrees
Regridder built


In [103]:
models_to_eval['HRES']

<xarray.DataArray 'tp' (time: 28, lat: 181, lon: 360)> Size: 7MB
array([[[2.28881836e-05, 2.28881836e-05, 2.28881836e-05, ...,
         2.28881836e-05, 2.28881836e-05, 2.28881836e-05],
        [8.77380371e-05, 8.77380371e-05, 8.77380371e-05, ...,
         6.86645508e-05, 8.77380371e-05, 8.77380371e-05],
        [2.36511230e-04, 2.36511230e-04, 2.47955322e-04, ...,
         2.25067139e-04, 2.25067139e-04, 2.36511230e-04],
        ...,
        [4.19616699e-05, 4.19616699e-05, 3.81469727e-05, ...,
         3.81469727e-05, 3.81469727e-05, 4.19616699e-05],
        [1.25885010e-04, 1.25885010e-04, 1.25885010e-04, ...,
         1.18255615e-04, 1.25885010e-04, 1.25885010e-04],
        [2.17437744e-04, 2.17437744e-04, 2.17437744e-04, ...,
         2.17437744e-04, 2.17437744e-04, 2.17437744e-04]],

       [[3.43322754e-05, 3.43322754e-05, 3.43322754e-05, ...,
         3.43322754e-05, 3.43322754e-05, 3.43322754e-05],
        [1.56402588e-04, 1.56402588e-04, 1.56402588e-04, ...,
         1.33514404e-04, 1.56402588e-04, 1.56402588e-04],
        [3.89099121e-04, 3.89099121e-04, 4.08172607e-04, ...,
         3.77655029e-04, 3.77655029e-04, 3.89099121e-04],
...
        [9.53674316e-03, 9.53674316e-03, 1.01470947e-02, ...,
         9.29260254e-03, 9.29260254e-03, 9.53674316e-03],
        [1.38702393e-02, 1.38702393e-02, 1.38702393e-02, ...,
         1.36566162e-02, 1.38702393e-02, 1.38702393e-02],
        [1.06658936e-02, 1.06658936e-02, 1.06658936e-02, ...,
         1.06658936e-02, 1.06658936e-02, 1.06658936e-02]],

       [[4.88281250e-04, 4.88281250e-04, 4.88281250e-04, ...,
         4.88281250e-04, 4.88281250e-04, 4.88281250e-04],
        [5.64575195e-04, 5.64575195e-04, 5.64575195e-04, ...,
         5.49316406e-04, 5.64575195e-04, 5.64575195e-04],
        [7.17163086e-04, 7.17163086e-04, 7.47680664e-04, ...,
         6.86645508e-04, 6.86645508e-04, 7.17163086e-04],
        ...,
        [9.56726074e-03, 9.56726074e-03, 1.01623535e-02, ...,
         9.30786133e-03, 9.30786133e-03, 9.56726074e-03],
        [1.39617920e-02, 1.39617920e-02, 1.39617920e-02, ...,
         1.37634277e-02, 1.39617920e-02, 1.39617920e-02],
        [1.07727051e-02, 1.07727051e-02, 1.07727051e-02, ...,
         1.07727051e-02, 1.07727051e-02, 1.07727051e-02]]], dtype=float32)
Coordinates:
    number      int64 8B 0
    surface     float64 8B 0.0
    valid_time  (time) datetime64[ns] 224B 2024-08-01T06:00:00 ... 2024-08-08
  * lat         (lat) float64 1kB -90.0 -89.0 -88.0 -87.0 ... 88.0 89.0 90.0
  * lon         (lon) float64 3kB 0.0 1.0 2.0 3.0 ... 356.0 357.0 358.0 359.0
  * time        (time) timedelta64[ns] 224B 0 days 06:00:00 ... 7 days 00:00:00
    expver      (time) <U4 448B '0001' '0001' '0001' ... '0001' '0001' '0001'
Attributes: (12/23)
    GRIB_paramId:                    228
    GRIB_dataType:                   fc
    GRIB_numberOfPoints:             6599680
    GRIB_typeOfLevel:                surface
    GRIB_stepUnits:                  1
    GRIB_stepType:                   instant
    ...                              ...
    GRIB_totalNumber:                0
    GRIB_units:                      m
    long_name:                       Total precipitation
    units:                           m
    standard_name:                   unknown
    regrid_method:                   nearest_s2d

In [131]:
for model_name, predictions in tqdm(models_to_eval.items(), desc="MSE Calc"):
    print(list(map(lambda x: x / 3.6e+12, predictions.time.values)))


MSE Calc: 100%|██████████| 4/4 [00:00<00:00, 3554.49it/s]

[numpy.timedelta64(6,'ns'), numpy.timedelta64(12,'ns'), numpy.timedelta64(18,'ns'), numpy.timedelta64(24,'ns'), numpy.timedelta64(30,'ns'), numpy.timedelta64(36,'ns'), numpy.timedelta64(42,'ns'), numpy.timedelta64(48,'ns'), numpy.timedelta64(54,'ns'), numpy.timedelta64(60,'ns'), numpy.timedelta64(66,'ns'), numpy.timedelta64(72,'ns'), numpy.timedelta64(78,'ns'), numpy.timedelta64(84,'ns'), numpy.timedelta64(90,'ns'), numpy.timedelta64(96,'ns'), numpy.timedelta64(102,'ns'), numpy.timedelta64(108,'ns'), numpy.timedelta64(114,'ns'), numpy.timedelta64(120,'ns'), numpy.timedelta64(126,'ns'), numpy.timedelta64(132,'ns'), numpy.timedelta64(138,'ns'), numpy.timedelta64(144,'ns'), numpy.timedelta64(150,'ns'), numpy.timedelta64(156,'ns'), numpy.timedelta64(162,'ns'), numpy.timedelta64(168,'ns')]
[numpy.timedelta64(6,'ns'), numpy.timedelta64(12,'ns'), numpy.timedelta64(18,'ns'), numpy.timedelta64(24,'ns'), numpy.timedelta64(30,'ns'), numpy.timedelta64(36,'ns'), numpy.timedelta64(42,'ns'), numpy.ti

In [138]:
all_results = []

In [139]:

# 5. Calculate MSE for each model and forecast horizon
for model_name, predictions in tqdm(models_to_eval.items(), desc="MSE Calc"):
    print(model_name)
    if eval_vars in predictions:
        pred_var = predictions[eval_vars]
    else:
        pred_var = predictions
    # else:
    #     logging.warning(f"'{eval_vars}' not found in predictions for model {model_name}. Skipping.")
    #     continue

    times = pred_var.time.values
        
    # for horizon_times, timestep in forecast_horizons_def.items():
    for timestep in times:
        # Slice predictions and targets to the current forecast horizon
        pred_sliced = pred_var.sel(time=timestep)
        targ_sliced = ground_truth_var.sel(time=timestep)
        
        # TODO: But we want the mse to be found at one particular time right
        # Calculate MSE over lat, lon, and time for the entire horizon
        mse = float(((pred_sliced - targ_sliced)**2).mean())

        # Store the result
        all_results.append({
            'init_date': init_date.strftime('%Y-%m-%d %H:%M:%S'),
            'forecast_horizon_times': timestep,
            'model': model_name,
            'mse': mse
        })
        logging.debug(f"Result: {init_date}, {model_name}, {timestep}-hour MSE = {mse:.6f}")


MSE Calc:  25%|██▌       | 1/4 [00:00<00:00,  4.93it/s]

Graphcast_Base


MSE Calc:  50%|█████     | 2/4 [00:00<00:00,  5.20it/s]

Graphcast_Finetuned1
Graphcast_Finetuned2


MSE Calc: 100%|██████████| 4/4 [00:00<00:00,  5.99it/s]

HRES


In [140]:
# 6. Save all results to a CSV file
logging.info("Evaluation loop finished. Saving results to CSV.")
results_df = pd.DataFrame(all_results)
set(results_df['model'])

2025-07-15 22:55:14,470 - INFO - Evaluation loop finished. Saving results to CSV.


{'Graphcast_Base', 'Graphcast_Finetuned1', 'Graphcast_Finetuned2', 'HRES'}

In [142]:
results_df

,init_date,forecast_horizon_times,model,mse
0,2024-08-01 00:00:00,0 days 06:00:00,Graphcast_Base,0.000002
1,2024-08-01 00:00:00,0 days 12:00:00,Graphcast_Base,0.000003
2,2024-08-01 00:00:00,0 days 18:00:00,Graphcast_Base,0.000003
3,2024-08-01 00:00:00,1 days 00:00:00,Graphcast_Base,0.000004
4,2024-08-01 00:00:00,1 days 06:00:00,Graphcast_Base,0.000003
...,...,...,...,...
107,2024-08-01 00:00:00,6 days 00:00:00,HRES,0.000761
108,2024-08-01 00:00:00,6 days 06:00:00,HRES,0.000814
109,2024-08-01 00:00:00,6 days 12:00:00,HRES,0.000873
110,2024-08-01 00:00:00,6 days 18:00:00,HRES,0.000935
